# Maximum Projection Image

-----
### Import dependancies

In [2]:
import os 

import tkinter as tk 
from tkinter import filedialog

import numpy as np 
import tifffile as tf

import aicsimageio

import czifile as czi
from lxml import etree
from tqdm import tqdm

# import napari

--------
### Get a list of .czi files

In [3]:
def get_image_files(folder):
    '''Loop though all file and folders to extract tif files.'''
    
    # initalise
    czi_file_list = []
    
    # For all subfolders and files. 
    for dirpath, dirnames, filenames in os.walk(folder):
        # For all files. 
        for file in filenames:
            # If it's a tiff file save it. 
            if file[-4:] == '.czi':
                czi_file_list.append(dirpath + '/' +file)
    
    # return file list. 
    return(czi_file_list)

----
### Get image file

In [4]:
# # # Creates dialogue to ask directory
# # # Get the folder containing the image stack. 
root = tk.Tk()
root.attributes('-topmost', True)
root.withdraw() # Stops a second window opening
top_folder = filedialog.askdirectory(title = 'Select Stack file')

print(top_folder)

N:/SHARED/1 Week/Ryan Corbyn/from XR/XR 199


In [5]:
folder_name = os.path.basename(top_folder)
os.path.dirname(top_folder) + '/' + folder_name + '_MIP_Images/'

'N:/SHARED/1 Week/Ryan Corbyn/from XR/XR 199_MIP_Images/'

---- 
### Create MIP for all files

In [ ]:
# Get all image files
image_files = get_image_files(top_folder)
folder_name = os.path.basename(top_folder)
# Create a new folder
save_folder = os.path.dirname(top_folder) + '/' + folder_name + '_MIP_Images/'
if os.path.exists(save_folder) == False: 
    os.makedirs(save_folder)

# Loop around all files
for i in tqdm( range( len(image_files) ) ):
    # Open the ith file
    single_file = image_files[i]
    # Use Aicsimageio to open image
    image = aicsimageio.AICSImage(single_file)
    # image_data = image.data()
    # Get the physical pixel sizes for the image
    image_resolution = image.physical_pixel_sizes
    # Initialise the maximum projection array
    max_I_projection = np.zeros((image.shape[0], 
                            image.shape[3],
                            image.shape[4]))

    # Loop around all time points
    for j in range(image.shape[0]):
        # Get the signle channel data for a 
        # single time point, Z, Y, X
        single_C_and_T = image.data[j, 3, :, :, :]
        # Perform maximum projection. 
        max_project = np.max(single_C_and_T, axis = 0)
        # Add the timepoint to the MIP array.
        max_I_projection[j, :, :] = max_project

    # Save the image to a .tif file
    tf.imwrite(save_folder + '/' + os.path.basename(single_file)[0:-4] + '.tif', 
                max_I_projection,
                resolution=(1/image_resolution[2], 1/image_resolution[1], 'MICROMETER'),
                # resolutionunit='MICROMETER',
                metadata={
                'unit': 'um',
                'axes': 'TYX'} )


 17%|█████████████▊                                                                     | 2/12 [01:32<07:40, 46.00s/it]